# 🎨 AI 캐리커처 서버 (행사 당일용)

부스 노트북에는 GPU가 없어서, **이 Colab의 GPU가 그림을 그리고** 부스 앱은 사진만 보내 결과를 받아간다.

**쓰는 법**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 확인
2. **런타임 → 모두 실행** (준비 4~6분)
3. 3번 셀 아래에 뜨는 `https://….trycloudflare.com` 주소를 복사
4. 부스 앱 첫 화면 **⚙ AI 서버 연결** → 주소 붙여넣기 → "연결 성공" 확인
5. 행사 내내 **이 크롬 탭을 닫지 말 것**. 맨 아래 5번 셀이 계속 돌아가는 게 정상이다 (서버를 켜 두는 역할).
   탭을 닫거나 Colab이 끊기면 서버가 꺼지지만, 부스는 물감 일러스트로 자동 전환되어 멈추지 않는다.

**폰으로 찍기**: 부스 앱 촬영 화면의 **📱 폰으로 찍기**를 누르면 QR이 뜬다. 폰으로 QR을 열어 찍으면 사진이 부스 화면에 자동으로 뜬다 (폰과 부스 PC가 다른 와이파이여도 됨).

무료 Colab은 GPU 사용 시간이 정해져 있다. 전날엔 쓰지 말고 아껴두고, 팀원 계정으로 예비 노트북을 하나 더 준비해 두자.

In [ ]:
#@title 1) 모델 올리기 (그림 생성 모델 + 얼굴 특징 IP-Adapter + 윤곽 고정 ControlNet)
!pip -q install diffusers transformers accelerate safetensors fastapi uvicorn 2>/dev/null
import torch, io, threading, time, subprocess, re, os
import numpy as np, cv2
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline, StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, DPMSolverMultistepScheduler
assert torch.cuda.is_available(), "GPU가 없어요 → 런타임 유형을 T4 GPU로 바꾸세요"
base = StableDiffusionImg2ImgPipeline.from_pretrained("dreamlike-art/dreamlike-anime-1.0", torch_dtype=torch.float16, safety_checker=None).to("cuda")
base.scheduler = DPMSolverMultistepScheduler.from_config(base.scheduler.config)
base.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-plus-face_sd15.safetensors")       # 얼굴 특징을 그림에 반영
cn = ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_canny", torch_dtype=torch.float16)                  # 사진의 윤곽선대로 그리게 붙잡는다
pipe = StableDiffusionControlNetImg2ImgPipeline(vae=base.vae, text_encoder=base.text_encoder, tokenizer=base.tokenizer, unet=base.unet, controlnet=cn, scheduler=base.scheduler, safety_checker=None, feature_extractor=base.feature_extractor, image_encoder=base.image_encoder, requires_safety_checker=False).to("cuda")
print("모델 준비 완료:", torch.cuda.get_device_name(0))

In [ ]:
#@title 2) 그리는 함수 (프롬프트·강도는 여기서 조절)
BASE = "caricature portrait drawing of a korean teenager, hand drawn by a street caricature artist, faithful likeness, distinctive facial features, big head, small shoulders, clean bold ink outline, soft marker coloring, warm healthy skin tone, natural black hair, plain white paper background, funny, best quality"
LIKE = ", natural eye size"                       # 미화 끔: 생긴 그대로
PRETTY = ", good looking, cute, rosy cheeks"      # 미화 켬: 살짝 예쁘게
NEG = "photo, realistic, 3d, blurry, lowres, deformed, ugly, extra fingers, text, watermark, signature, nsfw, multiple people, pens, markers, pencils, art supplies, frame, dark background, blue hair, purple hair, colorful hair, grey skin, pale skin, harsh shadow on face"
NEG_LIKE = ", anime, huge eyes, sparkling eyes, idol"     # 미화 끔일 때 더 피할 것
def prompts(beauty, glasses):
    # 모델은 77토큰까지만 읽고 뒤는 버린다. 지금 길이가 거의 한계라서, 말을 더하려면 다른 말을 빼야 한다 (넘으면 아래에서 알려줌)
    return BASE + (PRETTY if beauty else LIKE) + (", wearing glasses" if glasses else ""), NEG + ("" if beauty else NEG_LIKE)
STEPS = 24         #@param {type:"integer"}
IP_LIKE = 0.85     #@param {type:"number"}
IP_PRETTY = 0.75   #@param {type:"number"}
OUTLINE = 0.7      #@param {type:"number"}
HOLD = 0.5         #@param {type:"number"}
INK = 0.7          #@param {type:"number"}
HIRES = 768        #@param {type:"integer"}
PAPER = np.array([252, 249, 243], np.float32)    # 종이 색
lock = threading.Lock()
def split(img):
    # 부스가 보낸 사진에 투명도(사람 영역)가 있으면 떼어낸다
    m = np.asarray(img.getchannel("A").resize((512, 512), Image.LANCZOS)).astype(np.float32) / 255 if img.mode in ("RGBA", "LA") else None
    return img.convert("RGB").resize((512, 512), Image.LANCZOS), m
def edges(img, lo=50, hi=130):
    # 사진의 윤곽선(눈·코·입·턱선·머리 모양·안경테). ControlNet이 이 선을 따라 그리므로 표정과 생김새가 유지된다
    g = cv2.GaussianBlur(cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2GRAY), (0, 0), 1.0)
    return Image.fromarray(np.stack([cv2.Canny(g, lo, hi)] * 3, -1))
def keep_color(out, src, m, amount=0.9, amount_l=0.5):
    # 그림과 사진의 위치가 맞으므로, 넓은 범위의 색만 사진에서 가져온다 → 피부색·머리색·옷색이 실제와 같아진다
    z = out.size[0] / 512
    src = src.resize(out.size, Image.LANCZOS)
    o = cv2.cvtColor(np.asarray(out), cv2.COLOR_RGB2LAB).astype(np.float32); s = cv2.cvtColor(np.asarray(src), cv2.COLOR_RGB2LAB).astype(np.float32)
    m = (np.abs(s - s[:8, :8].reshape(-1, 3).mean(0)).sum(2) > 24).astype(np.float32) if m is None else cv2.resize(m, out.size, interpolation=cv2.INTER_LINEAR)
    k = cv2.GaussianBlur(cv2.erode(m, np.ones((int(5 * z) | 1,) * 2, np.uint8)), (0, 0), 4 * z)[..., None]
    d = cv2.GaussianBlur(s, (0, 0), 12 * z) - cv2.GaussianBlur(o, (0, 0), 12 * z)
    d[..., 0] *= amount_l; d[..., 1:] *= amount
    return Image.fromarray(cv2.cvtColor(np.clip(o + d * k, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB))
def ink(out, k, gain=14):
    # 주변보다 어두운 가는 선(윤곽선)을 더 까맣게 → 펜으로 그린 느낌
    z = out.size[0] / 512
    g = cv2.cvtColor(np.asarray(out), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255
    d = cv2.GaussianBlur(g, (0, 0), 1.0 * z) - cv2.GaussianBlur(g, (0, 0), 2.4 * z)
    line = cv2.GaussianBlur(np.clip(-d * gain, 0, 1), (0, 0), 0.5 * z)[..., None]
    return Image.fromarray(np.clip(np.asarray(out).astype(np.float32) * (1 - k * line), 0, 255).astype(np.uint8))
def paper(out, m):
    # 사람 바깥은 깨끗한 종이 색으로 (모델이 배경에 그린 얼룩·네모 테두리 제거)
    z = out.size[0] / 512
    m = cv2.resize(m, out.size, interpolation=cv2.INTER_LINEAR)
    k = cv2.GaussianBlur(cv2.dilate(m, np.ones((int(25 * z) | 1,) * 2, np.uint8)), (0, 0), 6 * z)[..., None]
    return Image.fromarray((np.asarray(out).astype(np.float32) * k + PAPER * (1 - k)).astype(np.uint8))
def draw(img, strength=1.5, beauty=0, glasses=0, seed=7):
    # 부스의 과장 강도(1.0~2.4) → 다시 그리는 정도(0.86~0.95). 앞쪽 절반 동안 윤곽선을 붙잡고(ControlNet) 뒤쪽 절반은 자유롭게 그리게 해서, 닮았으면서도 손그림 같은 굵은 선이 나온다
    s = min(0.95, max(0.86, 0.86 + (float(strength) - 1.0) * 0.065))
    img, m = split(img)
    prompt, neg = prompts(beauty, glasses)
    with lock:                                   # GPU는 한 번에 한 장
        pipe.set_ip_adapter_scale(IP_PRETTY if beauty else IP_LIKE)
        g = torch.Generator("cuda").manual_seed(seed)
        out = pipe(prompt, image=img, control_image=edges(img), ip_adapter_image=img, negative_prompt=neg, strength=s, controlnet_conditioning_scale=OUTLINE, control_guidance_end=HOLD, guidance_scale=7.0, num_inference_steps=STEPS, generator=g).images[0]
        if HIRES > 512:                          # 2단계: 크게 키운 그림을 살짝만 다시 그려서 선과 디테일을 또렷하게 (인쇄용)
            big = keep_color(out, img, m).resize((HIRES, HIRES), Image.LANCZOS)
            out = pipe(prompt, image=big, control_image=edges(big), ip_adapter_image=img, negative_prompt=neg, strength=0.35, controlnet_conditioning_scale=0.6, guidance_scale=6.0, num_inference_steps=STEPS, generator=g).images[0]
    out = keep_color(out, img, m)
    if INK > 0: out = ink(out, INK)
    return paper(out, m) if m is not None else out
for b_ in (0, 1):
    for t_ in prompts(b_, 1): assert len(base.tokenizer(t_).input_ids) <= 77, "프롬프트가 너무 길어요(77토큰 초과). 뒤쪽이 잘리니 줄이세요: " + t_
t = time.time(); _ = draw(Image.new("RGB", (512, 512), (230, 220, 210))); print("시험 그리기 %.1f초" % (time.time() - t))

In [ ]:
#@title 3) 서버 켜기 + 주소 만들기 (아래에 뜨는 주소를 부스 앱에 붙여넣기)
import uvicorn
from fastapi import FastAPI, Request, Response
from fastapi.responses import HTMLResponse
from starlette.concurrency import run_in_threadpool
app = FastAPI()
COUNT = {"n": 0, "fail": 0}
PHONE = {"id": 0, "data": None, "t": 0}          # 폰에서 찍어 보낸 사진 (부스가 가져가면 바로 지운다)

@app.get("/health")
def health(): return {"ok": True, "version": 3, "gpu": torch.cuda.get_device_name(0), "count": COUNT["n"]}

@app.post("/caricature")
async def caricature(req: Request, strength: float = 1.5, beauty: int = 0, glasses: int = 0):
    try:
        img = Image.open(io.BytesIO(await req.body()))
        out = await run_in_threadpool(draw, img, strength, beauty, glasses); COUNT["n"] += 1      # 그리는 동안에도 서버가 다른 요청(상태 확인·폰 사진)에 답하도록 따로 돌린다
    except Exception as e:                        # 한 장이 실패해도 서버는 계속 돈다 (부스가 다시 시도)
        COUNT["fail"] += 1; torch.cuda.empty_cache()
        return Response(str(e)[:200], status_code=500)
    buf = io.BytesIO(); out.save(buf, "JPEG", quality=93)
    return Response(buf.getvalue(), media_type="image/jpeg")

PHONE_PAGE = """<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>캐리커처 부스 · 폰으로 찍기</title>
<style>
body{margin:0;font-family:system-ui,sans-serif;background:#0b1a6e;color:#fff;text-align:center;padding:28px 16px}
h1{font-size:24px;margin:6px 0 4px}p{opacity:.85;margin:6px 0 20px;line-height:1.5}
label{display:block;max-width:420px;margin:0 auto 14px;border-radius:18px;font-weight:800}
.main{background:#ffd84d;color:#0b1a6e;font-size:26px;padding:24px}
.alt{border:2px solid rgba(255,255,255,.5);font-size:17px;padding:13px}
input{display:none}#pv{max-width:min(420px,100%);border-radius:14px;margin-top:10px;display:none}
#msg{font-size:19px;font-weight:700;min-height:30px;margin:12px 0}
</style></head><body>
<h1>캐리커처 부스</h1><p>얼굴이 화면에 크게 나오게,<br>정면에서 밝은 곳을 보고 찍어주세요</p>
<label class="main">📷 사진 찍기<input type="file" id="cam" accept="image/*" capture="environment"></label>
<label class="alt">🤳 셀카로 찍기<input type="file" id="selfie" accept="image/*" capture="user"></label>
<label class="alt">앨범에서 고르기<input type="file" id="pick" accept="image/*"></label>
<div id="msg"></div><img id="pv" alt="">
<script>
const msg=document.getElementById('msg'),pv=document.getElementById('pv');
async function send(f){
  if(!f)return; msg.textContent='보내는 중…';
  try{
    const img=new Image(); img.src=URL.createObjectURL(f); await img.decode();
    const k=Math.min(1,1600/Math.max(img.naturalWidth,img.naturalHeight));
    const c=document.createElement('canvas'); c.width=Math.round(img.naturalWidth*k); c.height=Math.round(img.naturalHeight*k);
    c.getContext('2d').drawImage(img,0,0,c.width,c.height);
    const blob=await new Promise(r=>c.toBlob(r,'image/jpeg',0.9));
    const res=await fetch('/phone/upload',{method:'POST',headers:{'Content-Type':'image/jpeg'},body:blob});
    if(!res.ok)throw new Error(res.status);
    pv.src=c.toDataURL('image/jpeg',0.5); pv.style.display='inline';
    msg.textContent='✅ 보냈어요! 부스 화면을 확인하세요';
  }catch(e){msg.textContent='❌ 전송 실패. 다시 시도해 주세요';}
}
for(const id of ['cam','selfie','pick'])document.getElementById(id).onchange=e=>{send(e.target.files[0]);e.target.value='';};
</script></body></html>"""

@app.get("/phone", response_class=HTMLResponse)
def phone_page(): return PHONE_PAGE

@app.post("/phone/upload")
async def phone_upload(req: Request):
    data = await req.body()
    if not data or len(data) > 8_000_000: return Response("사진이 없거나 너무 커요", status_code=400)
    PHONE.update(id=PHONE["id"] + 1, data=data, t=time.time())
    return {"ok": True, "id": PHONE["id"]}

@app.get("/phone/state")
def phone_state(): return {"id": PHONE["id"]}

@app.get("/phone/latest")
def phone_latest(after: int = 0):
    if PHONE["data"] is not None and time.time() - PHONE["t"] > 120: PHONE["data"] = None   # 2분 넘게 안 가져가면 버린다
    if PHONE["data"] is None or PHONE["id"] <= after: return Response(status_code=204)
    data, PHONE["data"] = PHONE["data"], None
    return Response(data, media_type="image/jpeg", headers={"X-Photo-Id": str(PHONE["id"])})

if "server" in globals(): server.should_exit = True; time.sleep(1.5)       # 이 셀을 다시 실행하면 서버도 새로 켠다
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared; chmod +x /content/cloudflared
subprocess.run("pkill -f cloudflared", shell=True)
proc = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, start_new_session=True)   # 셀을 ■로 멈춰도 주소 연결은 안 끊기게
URL = None
for line in proc.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: URL = m.group(0); break
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()   # 로그 비우기
print("\n" + "=" * 60 + "\n부스 앱 ⚙ AI 서버 연결에 붙여넣을 주소:\n\n   " + str(URL) + "\n\n" + "=" * 60)

In [ ]:
#@title 4) 확인: 주소로 직접 한 장 그려보기
import requests
for _ in range(30):                              # 새 주소가 인터넷에 퍼질 때까지 최대 2분 기다린다
    try: print(requests.get(URL + "/health", timeout=10).json()); break
    except Exception: time.sleep(4)
else: raise SystemExit("주소가 아직 안 열렸어요 → 1분 뒤 이 셀만 다시 실행")
buf = io.BytesIO(); Image.new("RGB", (512, 512), (200, 180, 170)).save(buf, "JPEG")
t = time.time(); r = requests.post(URL + "/caricature?strength=1.5", data=buf.getvalue(), timeout=120)
print("주소로 그리기", r.status_code, "%.1f초" % (time.time() - t), len(r.content), "bytes")

In [ ]:
#@title 5) 서버 유지 — 이 셀이 계속 돌아가는 게 정상 (끝낼 때만 ■ 정지)
from IPython.display import clear_output
while True:                                      # 셀이 돌고 있어야 Colab이 '사용 중'으로 보고 런타임을 끊지 않는다
    clear_output(wait=True)
    now = time.strftime("%H:%M", time.gmtime(time.time() + 9 * 3600))
    if proc.poll() is None: print("[%s] 서버 켜짐 · 그린 그림 %d장 (실패 %d)\n주소: %s" % (now, COUNT["n"], COUNT["fail"], URL))
    else: print("[%s] ⚠ 주소 연결이 끊겼어요 → ■로 이 셀을 멈추고 3번 셀부터 다시 실행한 뒤, 새 주소를 부스 앱에 넣으세요" % now)
    time.sleep(30)